# Cataloging nuclear data libraries

<!-- index:description -->
Build a reference DataFrame cataloging material, MT numbers, and nuclear-data libraries.
<!-- /index:description -->

<!-- generated:structure -->
```text
Cataloging nuclear data libraries
├── 1. Read `cross_sections.xml` from each library
├── 2. Extract MT numbers from HDF5 files
├── 3. Rearrange dataframe to one MT per row
├── 4. Build materials-MT-libraries catalog
└── 5. Save df and catalog for future reference
```
<!-- /generated:structure -->


## 1. Read `cross_sections.xml` from each library

In [1]:
import h5py
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

from openmc_workbench_config import DATA_ROOT

rows = []

for neutron_dir in sorted(DATA_ROOT.glob('*/neutron')):
    library = neutron_dir.parent.name
    xs_xml = neutron_dir.parent / 'cross_sections.xml'
    root = ET.parse(xs_xml).getroot()

    for element in root.findall('library'):
        if element.get('type') != 'neutron':
            continue

        rows.append({
            'materials': element.attrib['materials'],
            'path': Path(library) / element.attrib['path'],
            'library': library,
        })

df = pd.DataFrame(rows)
df

,materials,path,library
0,H1,endf-b8.0/neutron/H1.h5,endf-b8.0
1,H2,endf-b8.0/neutron/H2.h5,endf-b8.0
2,H3,endf-b8.0/neutron/H3.h5,endf-b8.0
3,He3,endf-b8.0/neutron/He3.h5,endf-b8.0
4,He4,endf-b8.0/neutron/He4.h5,endf-b8.0
...,...,...,...
2688,Es255,jendl-5/neutron/Es255.h5,jendl-5
2689,Fm252,jendl-5/neutron/Fm252.h5,jendl-5
2690,Fm253,jendl-5/neutron/Fm253.h5,jendl-5
2691,Fm255,jendl-5/neutron/Fm255.h5,jendl-5


## 2. Extract MT numbers from HDF5 files

In [2]:
def get_reactions(filename):
    with h5py.File(DATA_ROOT / filename) as ff:
        nuclide_name = list(ff.keys())[0]
        reactions = ff[nuclide_name]['reactions']

        mts = []
        for name in reactions:
            if name.startswith('reaction_'):
                mt = int(name.removeprefix('reaction_'))
                mts.append(mt)

    return sorted(mts)

df = df.assign(MTs=df['path'].map(get_reactions))
df

,materials,path,library,MTs
0,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,"[2, 102, 204, 301, 444, 901]"
1,H2,endf-b8.0/neutron/H2.h5,endf-b8.0,"[2, 16, 102, 203, 205, 301, 444, 901]"
2,H3,endf-b8.0/neutron/H3.h5,endf-b8.0,"[2, 16, 204, 301, 444, 901]"
3,He3,endf-b8.0/neutron/He3.h5,endf-b8.0,"[2, 102, 103, 104, 203, 204, 205, 207, 301, 44..."
4,He4,endf-b8.0/neutron/He4.h5,endf-b8.0,"[2, 301, 444, 901]"
...,...,...,...,...
2688,Es255,jendl-5/neutron/Es255.h5,jendl-5,"[2, 16, 17, 18, 37, 51, 52, 91, 102, 301, 444,..."
2689,Fm252,jendl-5/neutron/Fm252.h5,jendl-5,"[2, 5, 16, 17, 18, 22, 24, 28, 32, 33, 41, 51,..."
2690,Fm253,jendl-5/neutron/Fm253.h5,jendl-5,"[2, 5, 16, 17, 18, 22, 24, 28, 32, 33, 37, 41,..."
2691,Fm255,jendl-5/neutron/Fm255.h5,jendl-5,"[2, 16, 17, 18, 37, 51, 52, 53, 54, 55, 56, 57..."


## 3. Rearrange dataframe to one MT per row

In [3]:
df = df.explode('MTs', ignore_index=True)
df = df.rename(columns={'MTs': 'MT'})
df['MT'] = df['MT'].astype(int)
assert not df.duplicated(['materials', 'MT', 'library']).any()
df

,materials,path,library,MT
0,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,2
1,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,102
2,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,204
3,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,301
4,H1,endf-b8.0/neutron/H1.h5,endf-b8.0,444
...,...,...,...,...
219209,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,750
219210,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,799
219211,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,800
219212,Fm257,jendl-5/neutron/Fm257.h5,jendl-5,849


## 4. Build materials-MT-libraries catalog

In [4]:
catalog = (
    df.groupby(['materials', 'MT'], as_index=False)
      .agg(libraries=('library', ', '.join))
)
catalog

,materials,MT,libraries
0,Ac225,2,"endf-b8.0, endf-b8.1, jeff-4, jendl-5"
1,Ac225,5,jeff-4
2,Ac225,11,jeff-4
3,Ac225,16,"endf-b8.0, endf-b8.1, jeff-4, jendl-5"
4,Ac225,17,"endf-b8.0, endf-b8.1, jeff-4, jendl-5"
...,...,...,...
125201,Zr97,825,jendl-5
125202,Zr97,826,jendl-5
125203,Zr97,827,jendl-5
125204,Zr97,849,jendl-5


## 5. Save df and catalog for future reference

In [5]:
df_to_save = df.copy()
df_to_save['path'] = df_to_save['path'].astype(str)
df_to_save.to_feather('0_2_catalog_nuclear_data_libraries_df.feather')

catalog.to_feather(
    '0_2_catalog_nuclear_data_libraries_catalog.feather'
)

`Feather` in context.
| Format | Size | Speed | Advantages / limitations |
|---|---|---|---|
| Parquet | Small | Fast | Preserves data types, compression, cross-language support |
| Pickle | Variable | Fast | Convenient Python serialization, but unsafe from untrusted sources |
| CSV | Larger | Slower | Human-readable, universal, loses type information |
| Feather | Small | Very fast | Good for quick DataFrame exchange |
| SQLite | Variable | Query-dependent | Useful for SQL searches without loading the full catalog |